In [14]:
import os 
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_contents,fetch_website_links
from openai import OpenAI, api_key

load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')
gemini_base_url = os.getenv('GEMINI_BASE_URL')
gemini_model = os.getenv('GOOGLE_MODEL')

if not api_key:
    print("No API key is found")

gemini_client =  OpenAI(base_url=gemini_base_url, api_key=api_key)

link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

def get_links_user_prompt(url):
    user_prompt = f"""
    Here is the list of links on the website {url} -
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.

    Links (some might be relative links):

    """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

def select_relevant_links(url):
    response = gemini_client.chat.completions.create(
        model=gemini_model,
        messages=[{'role': "system", "content": link_system_prompt},
        {'role': 'user', "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result =  response.choices[0].message.content
    links = json.loads(result)
    print(f'Found {len(links)} relevant links')
    return links

def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
    You are looking at a company called: {company_name}
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code blocks.\n\n
    """
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

def create_brochure(company_name, url):
    response = gemini_client.chat.completions.create(
        model=gemini_model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

create_brochure('Hugging Face','https://huggingface.co')


Found 1 relevant links


# Welcome to Hugging Face

## The AI Community Building the Future

Hugging Face is the ultimate home and collaboration platform for the machine learning community. We empower developers, researchers, and organizations to create, discover, and build better machine learning applications together. Whether you are looking to explore cutting-edge artificial intelligence or deploy robust enterprise solutions, Hugging Face is where the global AI ecosystem connects.

---

## What We Offer

Hugging Face serves as the central hub for open-source AI development, providing a comprehensive ecosystem of tools and resources:

* **Models:** Browse, test, and utilize over 2 million machine learning models, ranging from advanced language models and computer vision tools to audio and diffusion networks.
* **Datasets:** Access over 500,000 datasets to train, fine-tune, and evaluate your machine learning pipelines.
* **Spaces:** Explore and run over 1 million interactive AI applications, workflows, and demos directly in your browser.
* **Enterprise Solutions:** Scale your AI initiatives with Hugging Face PRO, dedicated enterprise support, custom inference providers, inference endpoints, and secure storage buckets.

---

## Company Culture & Community

At our core, Hugging Face is driven by a vibrant, open-source community spirit. Our culture thrives on transparency, collaborative research, and shared knowledge. 

* **Connect & Collaborate:** Engage with fellow innovators through our active community blog, daily research papers, Discord server, user forums, and GitHub repositories.
* **Inclusive Topics:** Dive into diverse domains including Natural Language Processing (NLP), Audio, Computer Vision (CV), Reinforcement Learning (RL), AI Ethics, and Game Development.

---

## Customers & Partners

Hugging Face proudly supports a vast spectrum of users—from independent developers and academic researchers to major global technology leaders and enterprise innovators. Organizations leverage our platform to streamline their machine learning workflows, host public and private models, and accelerate their journey into production-grade artificial intelligence.

---

## Careers & Growth

Are you passionate about shaping the future of artificial intelligence? Hugging Face is always looking for brilliant minds to join our mission. 

* **Shape the Cutting Edge:** Work alongside top-tier researchers and engineers on foundational technologies that power the global AI movement.
* **Join the Mission:** If you are dedicated to open collaboration, innovation, and pushing the boundaries of machine learning, check out our latest opportunities and become part of the team building the future.